# Butterfly DDPM — Evaluation

Toy-task FID for the butterfly stage. Mirrors `colab_faces_evaluate.ipynb`
but pulls real reference images from the Hugging Face butterfly dataset
(no held-out test_files.txt needed).

## 1 · Mount Drive and clone repo

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH   = "feat/eval"
WORKDIR  = Path("/content/Human-Faces-Generation-Diffusion-Models")

# Step out of WORKDIR before deleting it; otherwise re-running this cell raises
# `getcwd: cannot access parent directories` because the shell is still inside it.
os.chdir("/content")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"
os.chdir(WORKDIR)
print("Project root:", WORKDIR)

## 2 · Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q torch-fidelity

## 3 · Run config

In [ ]:
import json
from pathlib import Path

DRIVE_ROOT       = Path("/content/drive/MyDrive/aml")
RUNS_ROOT        = DRIVE_ROOT / "ddpm_runs"
RUN_NAME         = "butterfly_run_001"   # ← edit this for a different run
RUN_DIR          = RUNS_ROOT / RUN_NAME

CHECKPOINT_PATH  = RUN_DIR / "ddpm_final.pth"
RUN_CONFIG_PATH  = RUN_DIR / "run_config.json"

REAL_DIR         = RUN_DIR / "real_test_set"
FAKE_DIR         = RUN_DIR / "generated"
EVAL_DIR         = RUN_DIR / "eval"
FID_JSON_PATH    = RUN_DIR / "fid.json"

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint missing: {CHECKPOINT_PATH}")

run_config = {}
if RUN_CONFIG_PATH.is_file():
    run_config = json.loads(RUN_CONFIG_PATH.read_text())
model_cfg = run_config.get("model", {})

IMAGE_SIZE     = int(run_config.get("image_size", 128))
TIMESTEPS      = int(run_config.get("timesteps", 1000))
BASE_CHANNELS  = int(model_cfg.get("base_channels", 64))
TIME_DIM       = int(model_cfg.get("time_dim", 256))
BATCH_SIZE     = 16
NUM_IMAGES     = 100  # butterfly test split size from data/dataloader.py splits()

print(f"RUN={RUN_NAME}  image_size={IMAGE_SIZE}  base_channels={BASE_CHANNELS}")

## 4 · Verify GPU

In [ ]:
import torch
print("torch:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not active. Runtime → Change runtime type → GPU.")

## 5 · Generate butterfly images

Calls `scripts/generate_butterflies.py` to write individual PNGs into
`generated/`.

In [ ]:
FAKE_DIR.mkdir(parents=True, exist_ok=True)

!python3 scripts/generate_butterflies.py \
    --checkpoint    "$CHECKPOINT_PATH" \
    --out_dir       "$FAKE_DIR" \
    --num_images    $NUM_IMAGES \
    --batch_size    $BATCH_SIZE \
    --image_size    $IMAGE_SIZE \
    --timesteps     $TIMESTEPS \
    --base_channels $BASE_CHANNELS \
    --time_dim      $TIME_DIM \
    --device        cuda

## 6 · Export the butterfly test split as PNGs

In [ ]:
REAL_DIR.mkdir(parents=True, exist_ok=True)

!python3 scripts/export_butterfly_reference.py \
    --out-dir    "$REAL_DIR" \
    --split      test \
    --image-size $IMAGE_SIZE

## 7 · Compute FID and save grids

In [ ]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
GEN_GRID  = EVAL_DIR / "generated_grid.png"
REAL_GRID = EVAL_DIR / "real_grid.png"

!python3 scripts/evaluate_fid.py \
    --real-dir       "$REAL_DIR" \
    --fake-dir       "$FAKE_DIR" \
    --image-size     $IMAGE_SIZE \
    --run-name       "$RUN_NAME" \
    --out-json       "$FID_JSON_PATH" \
    --grid-out       "$GEN_GRID" \
    --real-grid-out  "$REAL_GRID"

## 8 · Show grids and FID

In [ ]:
from IPython.display import Image, display, Markdown

fid_payload = json.loads(FID_JSON_PATH.read_text())
display(Markdown(f"### FID for `{RUN_NAME}`: **{fid_payload['fid']:.4f}**"))
for grid in (GEN_GRID, REAL_GRID):
    if grid.is_file():
        display(Markdown(f"#### {grid.stem}"))
        display(Image(filename=str(grid)))

## 9 · Aggregate FID across butterfly runs

In [ ]:
SUMMARY_DIR = Path("results/butterflies/eval")
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = SUMMARY_DIR / "fid_summary.csv"
SUMMARY_MD  = SUMMARY_DIR / "fid_summary.md"

!python3 scripts/aggregate_fid.py \
    --runs-dir "$RUNS_ROOT" \
    --out-csv  "$SUMMARY_CSV" \
    --out-md   "$SUMMARY_MD" \
    --require-fid

from IPython.display import Markdown, display
if SUMMARY_MD.is_file():
    display(Markdown(SUMMARY_MD.read_text()))